# Запуск модели

Этот ноутбук позволяет запустить модель из нашего модуля transformer_lab

Устанавливаем наш пакет и импортируем все нужное

In [ ]:
%pip install -q git+https://github.com/tereshkovyv/transformer_lab

In [ ]:
import torch

from transformer_lab.model.gpt_model import GPTModel
from transformer_lab.model.generate import generate, generate_text_simple
from transformer_lab.training.data import Llama2Tokenizer

In [ ]:
CONFIG = {
    "vocab_size": 32000,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

torch.manual_seed(44)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPTModel(CONFIG).to(device)
model.eval()
print(device, sum(p.numel() for p in model.parameters()))

Импортируем веса из .pth файла:

In [ ]:
from google.colab import files
WEIGHTS_PATH = next(iter(files.upload()))
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
model.eval()
print("loaded", WEIGHTS_PATH)

Токенайзер - берем Llama 2 (SentencePiece BPE, 32 000), как в Эксперименте Дениса:

In [ ]:
tokenizer = Llama2Tokenizer()
idx = torch.tensor([tokenizer.encode("Every effort moves you")], device=device)
print(model(idx).shape)  # (1, 4, 32000)

Вариант 1: Жадно берем самый вероятный логит:

In [ ]:
out = generate_text_simple(model, idx, max_new_tokens=10, context_size=CONFIG["context_length"])
print(tokenizer.decode(out[0].tolist()))

Вариант 2: Используем температуру

In [ ]:
out = generate(model, idx, max_new_tokens=10, context_size=CONFIG["context_length"], temperature=1.0, top_k=25)
print(tokenizer.decode(out[0].tolist()))